In [1]:
import pandas as pd
import yfinance as yf
import time
import os
import re
from IPython.display import clear_output

In [2]:
#ADJ is the amount in dollars to make up for the price diff bw/ yfinance api and broker data
yfinance_sym_dic = { 
    'MNQ': {'SYM':'MNQ=F', 'ADJ': 0},
    'NQ': {'SYM':'NQ=F', 'ADJ': 0},
    'MES': {'SYM':'MES=F', 'ADJ': 0},
    'ES': {'SYM':'ES=F', 'ADJ': 0},
    'US100': {'SYM':'MNQ=F', 'ADJ': -53.18},
    'GC': {'SYM':'GC=F', 'ADJ': 0},
    'MGX': {'SYM':'MGC=F', 'ADJ': 0},
    'SI': {'SYM':'SI=F', 'ADJ': 0},
    'SIL': {'SYM':'SIL=F', 'ADJ': 0},
    'XAUUSD': {'SYM':'GC=F', 'ADJ': 0},
    'AGXUSD': {'SYM':'SI=F', 'ADJ': 0},
    'BZ': {'SYM':'BZ=F', 'ADJ': 0}, # Brent Crude Futures
    'CL': {'SYM':'CL=F', 'ADJ': 0}, # WTI Crude Futures
    'BTC': {'SYM':'BTC-USD', 'ADJ': 0},
    'ETH': {'SYM':'ETH-USD', 'ADJ': 0}
}


def get_live_price(ticker_symbol: str, yfinance_map: dict)-> float:
    # Initialize the Ticker object 
    if ticker_symbol in yfinance_map.keys():
        ticker = yf.Ticker(yfinance_map[ticker_symbol]['SYM'])
        # .fast_info provides the most recent 'last_price'
        # This is faster than fetching the full .info dictionary
        current_price = ticker.fast_info['last_price'] + yfinance_map[ticker_symbol]['ADJ']
    else:
        ticker = yf.Ticker(ticker_symbol)
        current_price = ticker.fast_info['last_price']
        
    return current_price

# Read the trades worksheet

In [3]:
# 1. Replace with your actual Google Sheet ID
# (Found in the URL: https://docs.google.com/spreadsheets/d/SHEET_ID/edit)
SHEET_ID = "1HJ9h7UEtUQCXNA58UkZyPsHogJWBAcB1lNWt9nOPMR4"

# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Trades'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
df = pd.read_csv(url)

# 5. Keep open trades only
df = df[df.Closed=='No'].copy()

# Fill numberic columns' NA with 0 and cast numeric columns from str to float type
cols = ['Open Price', 'Close Price', 'Commission','Risk ($)', 'Balance at Open', 'PnL']
df[cols] = df[cols].fillna('0')
for c in cols:
    df[c] = df[c].apply(lambda x: float(re.sub(r"\(", "-", re.sub(r"[,\)]", "", x))))
df.head()

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,PnL,Closed,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2,Unnamed: 20,Unnamed: 21
0,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34,221.02,216.30,0.00,59800.00,NaN,...,160.48,No,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN
1,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,COF,-34,220.62,216.30,0.00,42336.39,224.59,...,146.88,No,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN
2,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,CVNA,-140,72.32,74.54,-2.91,42336.39,81.13,...,-313.11,No,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN
3,8/25/2026 0:00:00,8/25/2026,Paper Trading #1,CVNA,-171,75.57,74.54,0.00,59313.65,NaN,...,176.13,No,0,NaN,#VALUE!,NaN,NaN,NaN,NaN,NaN
4,8/27/2026 7:02:24,8/27/2026,Paper Trading #2,CVNA,-800,75.45,74.54,0.00,100000.00,76.89,...,726.00,No,0,NaN,#VALUE!,NaN,NaN,NaN,NaN,NaN


# Get Point Values

In [4]:
# 2. Specify the tab name (optional, defaults to the first sheet)
SHEET_NAME = 'Symbols'

# 3. Construct the export URL
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"

# 4. Load into DataFrame
point_val_df = pd.read_csv(url, header=None, names=['Symbol', 'Point Value'])
point_val_df.head()

,Symbol,Point Value
0,ADA,1
1,COF,1
2,CVNA,1
3,MNQ,2
4,MES,5


# Get Prices

In [5]:
price_df = pd.DataFrame(df['Symbol']).drop_duplicates()
price_df['Current Price'] = price_df.Symbol.apply(lambda x : get_live_price(x, yfinance_sym_dic))
price_df

,Symbol,Current Price
0,COF,215.985001
2,CVNA,74.489998
5,MES,7735.250000


# Append Price to trades DF

In [6]:
df = pd.merge(df, price_df, on='Symbol', how='left')
df = pd.merge(df, point_val_df, on='Symbol', how='left')
df['Point Value'] = df['Point Value'].fillna(1)
df['PnL'] = (df['Volume'] * (df['Current Price']-df['Open Price']) * df['Point Value']).round(2)
df

,Open Time,Open Date,Account,Symbol,Volume,Open Price,Close Price,Commission,Balance at Open,Stop Loss,...,Is Closed,Close Time,Close Date,Entry Link,Exit Link 1,Exit Link 2,Unnamed: 20,Unnamed: 21,Current Price,Point Value
0,8/24/2026 0:00:00,8/24/2026,Paper Trading #1,COF,-34,221.02,216.30,0.00,59800.00,NaN,...,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN,215.985001,1
1,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,COF,-34,220.62,216.30,0.00,42336.39,224.59,...,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN,215.985001,1
2,8/24/2026 0:00:00,8/24/2026,Tradestation - Equity,CVNA,-140,72.32,74.54,-2.91,42336.39,81.13,...,0,NaN,#VALUE!,Link,NaN,NaN,NaN,NaN,74.489998,1
3,8/25/2026 0:00:00,8/25/2026,Paper Trading #1,CVNA,-171,75.57,74.54,0.00,59313.65,NaN,...,0,NaN,#VALUE!,NaN,NaN,NaN,NaN,NaN,74.489998,1
4,8/27/2026 7:02:24,8/27/2026,Paper Trading #2,CVNA,-800,75.45,74.54,0.00,100000.00,76.89,...,0,NaN,#VALUE!,NaN,NaN,NaN,NaN,NaN,74.489998,1
5,8/27/2026 8:39:41,8/27/2026,Paper Trading #2,MES,-6,7730.00,7719.09,0.00,0.00,7764.25,...,0,NaN,#VALUE!,NaN,NaN,NaN,NaN,NaN,7735.250000,5


# Group by account and symbol to report

In [7]:
print(df.groupby('Account').agg({'PnL': sum}))
print('\n', 50 * '-', '\n')
print(df.groupby('Symbol').agg({'Volume': sum, 'PnL': sum}))
print('\n', 50 * '-', '\n')
print(df.groupby(['Symbol','Account']).agg({'Volume': sum, 'PnL': sum}))
print('\n', 50 * '-', '\n')
print(df.groupby(['Account','Symbol']).agg({'Volume': sum, 'PnL': sum}))

                          PnL
Account                      
Paper Trading #1       355.87
Paper Trading #2       610.50
Tradestation - Equity -146.21

 -------------------------------------------------- 

        Volume     PnL
Symbol                
COF        -68  328.78
CVNA     -1111  648.88
MES         -6 -157.50

 -------------------------------------------------- 

                              Volume     PnL
Symbol Account                              
COF    Paper Trading #1          -34  171.19
       Tradestation - Equity     -34  157.59
CVNA   Paper Trading #1         -171  184.68
       Paper Trading #2         -800  768.00
       Tradestation - Equity    -140 -303.80
MES    Paper Trading #2           -6 -157.50

 -------------------------------------------------- 

                              Volume     PnL
Account               Symbol                
Paper Trading #1      COF        -34  171.19
                      CVNA      -171  184.68
Paper Trading #2      CVNA    